# CS604 Colab launcher — Ana: why the classes are hard (figures, no training)

Uses a holdout's errors and the checkpoint that made them (it never trained on those photos). `SOURCE` in the first cell picks which:
- `"pilot"` (default): the LoRA pilot from `colab_error-analysis_where-errors-come-from.ipynb` (`CS604_RUNS/error_analysis/errors_lora.csv`, 1,000 classes × 5 holdout photos);
- `"allclass"`: the all-class LoRA run (`errors_lora_all.csv`, 10,000 classes × 5), if its folders are still on Drive.

1. `scripts/confusion_pairs.py` ranks class pairs by how often the model confused them and writes per-class accuracy.
2. Two scripts choose which classes they need, so only those are fetched: `train.tar.gz` (40 GB) is read once and only the shortlisted classes are kept.
3. `scripts/showcase_classes.py` embeds every photo of the shortlisted classes with the checkpoint and draws the report figures:
   - `intra_class_variation.png`: **one species, different looks**: among the weakest classes, the ones whose photos split into two groups that are least alike;
   - `inter_class_similarity.png`: **different species, same look**: the confused pairs whose class prototypes are most alike;
   - `most_confused_pair.png`: the pair confused most often, with the holdout photos the model got wrong in **red**;
   - `showcase.json`: the numbers behind them, to quote in the report.
4. `scripts/pair_gallery.py` draws all the top confused pairs side by side (`pairs_01.png`, …), red frames on the mistakes.

Outputs in `CS604_RUNS/confused_pairs_<SOURCE>/` (pairs) and `CS604_RUNS/confused_pairs_<SOURCE>/showcase/` (figures).

About 20–30 min, almost all of it the download and one pass through the archive. **Any GPU works** (a T4 is plenty: only ~5,000 photos are embedded). Use **Runtime → Run all**.

In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
# Must match a folder (or shortcut) in My Drive — not the browser URL.
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "ana-branch"
# Which holdout to use: its errors file and the checkpoint that made them (it must not have
# trained on those holdout photos).
SOURCE = "pilot"
SOURCES = {
    # LoRA pilot (colab_error-analysis_where-errors-come-from.ipynb): 1,000 classes x 5 holdout photos
    "pilot": dict(errors=f"{DRIVE_OUT}/error_analysis/errors_lora.csv",
                  run_dir=f"{DRIVE_OUT}/ablation_ablation_train_crops/runs/lora_20261020_102852",
                  classes=1000),
    # All-class LoRA run (colab_lora-r24_336px_all-10000-classes_topk-and-knn.ipynb): 10,000 classes x 5
    "allclass": dict(errors=f"{DRIVE_OUT}/allclass_holdout_analysis/errors/errors_lora_all.csv",
                     run_dir=f"{DRIVE_OUT}/ablation_allclass_holdout_lora_no_extras/runs/A_all_classes_144min/lora_20261026_123804",
                     classes=10000),
}
ERRORS_CSV, RUN_DIR, HOLDOUT_CLASSES = (SOURCES[SOURCE][k] for k in ("errors", "run_dir", "classes"))
OUT_NAME = f"confused_pairs_{SOURCE}"
# Pair gallery
TOP_PAIRS = 30          # pairs from the top of confused_pairs.csv (most confusions first)
MUTUAL_ONLY = True      # only pairs confused in both directions: the clearest look-alikes
EXTRA_PAIRS = []        # pairs to add by hand, e.g. ["2099-2100", "319-318"]
PER_CLASS = 8           # photos per class row
# Report figures
WEAK_CLASSES = 60       # weakest classes searched for two different looks
SHOW_CLASSES = 2        # classes shown in intra_class_variation.png
SHOW_PAIRS = 3          # pairs shown in inter_class_similarity.png
# End the runtime after the last cell, or as soon as a step fails.
AUTO_DISCONNECT = True


def end_runtime(reason):
    """Flush Drive writes, then release the GPU so it stops using compute units."""
    print(reason, flush=True)
    if not AUTO_DISCONNECT:
        return
    from google.colab import drive, runtime
    try:
        drive.flush_and_unmount()
        print("Drive flushed and unmounted", flush=True)
    except Exception as error:
        print(f"Drive flush failed: {error}", flush=True)
    runtime.unassign()

## 1. Clone or pull the repo

In [ ]:
import os
from getpass import getpass

token = getpass("GitHub PAT (hidden; repo scope). Leave empty if the repo is public: ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    if token:
        !git -C {REPO_DIR} pull {clone_url} {BRANCH}
    else:
        !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {clone_url} {REPO_DIR}

%cd {REPO_DIR}
!git branch --show-current
!git rev-parse --short HEAD

## 2. Install requirements

In [ ]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])
assert Path(ERRORS_CSV).is_file(), (
    f"{ERRORS_CSV} not found. Upload the errors file to that folder in Drive "
    "(make the folders if needed), or fix SOURCES in the first cell.")
HAVE_CKPT = (Path(RUN_DIR) / "last.pt").is_file()
if not HAVE_CKPT:
    print(f"No checkpoint in {RUN_DIR}: the pair gallery still runs, the report figures are skipped. "
          "Set RUN_DIR to the run folder (with last.pt and config.yaml) to get them.")

## 4. Rank the confused pairs and shortlist the classes

Reads only the errors file, so this takes seconds.

In [ ]:
import subprocess
import sys

OUT = Path(DRIVE_OUT) / OUT_NAME
SHOW = OUT / "showcase"


def run(args, what):
    proc = subprocess.run([sys.executable, "-u", *args])
    if proc.returncode != 0:
        end_runtime(f"{what} failed (exit {proc.returncode})")
        raise SystemExit(f"{what} failed")


run(["scripts/confusion_pairs.py", "--errors", ERRORS_CSV, "--out_dir", str(OUT),
     "--num_classes", str(HOLDOUT_CLASSES)], "confusion pairs")
gallery_args = ["--pairs_csv", str(OUT / "confused_pairs.csv"), "--errors", ERRORS_CSV,
                "--out_dir", str(OUT), "--top", str(TOP_PAIRS), "--per_class", str(PER_CLASS)]
if MUTUAL_ONLY:
    gallery_args.append("--mutual_only")
if EXTRA_PAIRS:
    gallery_args += ["--pairs", *EXTRA_PAIRS]
show_args = ["--pairs_csv", str(OUT / "confused_pairs.csv"), "--class_accuracy", str(OUT / "class_accuracy.csv"),
             "--errors", ERRORS_CSV, "--out_dir", str(SHOW), "--weak", str(WEAK_CLASSES),
             "--top_pairs", str(TOP_PAIRS), "--show_classes", str(SHOW_CLASSES), "--show_pairs", str(SHOW_PAIRS)]
run(["scripts/pair_gallery.py", *gallery_args, "--classes_only"], "choosing pairs")
lists = [OUT / "classes.txt"]
if HAVE_CKPT:
    run(["scripts/showcase_classes.py", *show_args, "--classes_only"], "choosing classes")
    lists.append(SHOW / "classes.txt")
WANT = {int(c) for f in lists for c in f.read_text().split()}
print(f"{len(WANT)} classes to fetch")

## 5. Download the training archive and keep only those classes

`train.tar.gz` is one 40 GB archive, so it has to be downloaded whole; it is read once as a stream and only the wanted class folders are written to `/content/data/train/`. Rerunning skips classes already there.

In [ ]:
import tarfile

from huggingface_hub import hf_hub_download

REPO_ID = "doem1997/cs604-course-data"
data = Path(DATA_DIR)
(data / "train").mkdir(parents=True, exist_ok=True)
hf_hub_download(REPO_ID, "train.txt", repo_type="dataset", local_dir=DATA_DIR)
names = {f"class_{c:04d}" for c in WANT}
missing = {n for n in names if not any((data / "train" / n).glob("*"))}
if missing:
    archive = hf_hub_download(REPO_ID, "train.tar.gz", repo_type="dataset", local_dir="/content/raw")
    kept = 0
    with tarfile.open(archive, "r|gz") as tf:
        for m in tf:
            parts = m.name.split("/")
            if m.isfile() and len(parts) >= 2 and parts[-2] in missing:
                dest = data / "train" / parts[-2] / parts[-1]
                dest.parent.mkdir(parents=True, exist_ok=True)
                dest.write_bytes(tf.extractfile(m).read())
                kept += 1
    print(f"kept {kept} photos from {len(missing)} classes")
    Path(archive).unlink()  # free the 40 GB
# The scripts read paths from train.txt; check they point at the extracted photos
first = next(l.split()[0] for l in (data / "train.txt").read_text().splitlines()
             if l.split() and int(l.split()[1]) in WANT)
assert (data / first).is_file(), f"train.txt path {first} not found under {DATA_DIR}"
print("photos ready, e.g.", first)

## 6. Report figures (embeds the shortlisted classes on the GPU)

In [ ]:
if HAVE_CKPT:
    run(["scripts/showcase_classes.py", *show_args, "--run_dir", RUN_DIR, "--data_root", DATA_DIR], "report figures")
else:
    print("skipped: no checkpoint (see step 3)")

## 7. Pair gallery

In [ ]:
run(["scripts/pair_gallery.py", *gallery_args, "--data_root", DATA_DIR], "pair gallery")

## 8. Results

- **showcase.json:** `look_cosine` is how alike a class's two looks are; `prototype_cosine` is how alike two classes are. When a species' two looks are less alike than two different species (`for_scale`), one prototype per class cannot separate them.
- **Figures:** check they show what the title says before putting them in the report; if a class is a bad example (e.g. a mislabelled photo), raise `SHOW_CLASSES` / `SHOW_PAIRS` and pick from the next ones.
- **Pair gallery:** for each pair, do the two classes really look the same? Red frames are the photos the model got wrong. Look for genuinely identical species, a different life stage or part (caterpillar vs moth), a tiny subject, or a photo that looks mislabelled.

In [ ]:
import json

import pandas as pd
from IPython.display import Image, display

if (SHOW / "showcase.json").is_file():
    print(json.dumps(json.loads((SHOW / "showcase.json").read_text()), indent=2))
for name in ("intra_class_variation.png", "inter_class_similarity.png", "most_confused_pair.png"):
    if (SHOW / name).is_file():
        print(name)
        display(Image(filename=str(SHOW / name)))
print(json.dumps(json.loads((OUT / "summary.json").read_text()), indent=2))
display(pd.read_csv(OUT / "class_accuracy.csv").head(20))
display(pd.read_csv(OUT / "pairs_shown.csv"))
for png in sorted(OUT.glob("pairs_*.png")):
    print(png.name)
    display(Image(filename=str(png)))

## 9. End the runtime

In [ ]:
end_runtime("all cells finished; ending the runtime")